# 🔥 Meme Popularity Score Prediction

**Predicting how popular a meme will get, using engagement and posting metadata.**

### 📌 What this notebook covers
1. Importing libraries & loading the dataset
2. Exploratory Data Analysis (EDA)
3. Data Leakage Check
4. Data Preprocessing
5. Model Building (Random Forest & XGBoost)
6. Model Evaluation
7. Feature Importance
8. Conclusion

This notebook is written in a **beginner-friendly** way with explanations at every step. 🙂

## 1. Importing Libraries 📚

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

sns.set_style('whitegrid')
%matplotlib inline

## 2. Loading the Dataset 📂

In [ ]:
df = pd.read_csv('/kaggle/input/meme-popularity-dataset/meme_popularity_dataset_100000_rows.csv')
print('Shape of dataset:', df.shape)
df.head()

### Column meanings
- **likes, shares, comments, views** → engagement metrics on the meme post
- **watch_time_sec** → average time (in seconds) people spent watching the meme
- **caption_length** → number of characters in the caption
- **hashtags** → number of hashtags used
- **posting_hour** → hour of the day (0-23) the meme was posted
- **creator_followers** → number of followers the creator has
- **popularity_score** → target variable, meme's popularity score (0-100)

## 3. Basic Data Check ✅

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

**Good news:** there are no missing values in this dataset, so we don't need to handle any imputation. 👍

## 4. Exploratory Data Analysis 📊

### 4.1 Distribution of the Target Variable (popularity_score)

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['popularity_score'], bins=40, kde=True, color='orange')
plt.title('Distribution of Popularity Score')
plt.xlabel('Popularity Score')
plt.show()

Notice the spike near **100** — many memes hit the maximum possible score (the score seems to be capped at 100).

### 4.2 Correlation Heatmap

In [ ]:
plt.figure(figsize=(9,7))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

### 4.3 Relationship between Likes/Shares and Popularity Score

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
sns.scatterplot(x='likes', y='popularity_score', data=df.sample(3000, random_state=1), ax=axes[0], alpha=0.4)
axes[0].set_title('Likes vs Popularity Score')

sns.scatterplot(x='shares', y='popularity_score', data=df.sample(3000, random_state=1), ax=axes[1], alpha=0.4, color='green')
axes[1].set_title('Shares vs Popularity Score')
plt.show()

Both **likes** and **shares** show a positive but non-linear relationship with popularity score — this hints that tree-based models will work better than a simple straight-line (linear) model.

## 5. Data Leakage Check 🔍

Before modelling, it's important to check whether any single feature can **perfectly** predict the target (that would be data leakage, not real learning).

In [ ]:
correlations = df.corr(numeric_only=True)['popularity_score'].sort_values(ascending=False)
print(correlations)

**Observation:** the highest correlation (`likes` ≈ 0.55, `shares` ≈ 0.55) is moderate, not close to 1.0.
This confirms there is **no direct leakage** — `popularity_score` is not a simple formula of one column, so the model has to genuinely learn patterns.

## 6. Data Preprocessing 🛠️

In [ ]:
X = df.drop(columns=['popularity_score'])
y = df['popularity_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Training samples:', X_train.shape[0])
print('Testing samples :', X_test.shape[0])

All the columns are already numeric, so no encoding is required. This keeps preprocessing simple for beginners.

## 7. Model Building 🤖

### 7.1 Random Forest Regressor (Baseline Model)

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)

print('Random Forest Results')
print('R2 Score :', round(r2_score(y_test, rf_preds), 4))
print('MAE      :', round(mean_absolute_error(y_test, rf_preds), 3))
print('RMSE     :', round(np.sqrt(mean_squared_error(y_test, rf_preds)), 3))

### 7.2 XGBoost Regressor (Final Model)

In [ ]:
xgb_model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)

print('XGBoost Results')
print('R2 Score :', round(r2_score(y_test, xgb_preds), 4))
print('MAE      :', round(mean_absolute_error(y_test, xgb_preds), 3))
print('RMSE     :', round(np.sqrt(mean_squared_error(y_test, xgb_preds)), 3))

## 8. Model Evaluation 📈

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(y_test, xgb_preds, alpha=0.3, color='purple')
plt.plot([0,100], [0,100], color='red', linestyle='--')
plt.xlabel('Actual Popularity Score')
plt.ylabel('Predicted Popularity Score')
plt.title('Actual vs Predicted (XGBoost)')
plt.show()

The points lie very close to the red diagonal line, meaning our predictions are very close to the actual popularity scores. 🎯

## 9. Feature Importance 🌟

In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Feature Importance (XGBoost)')
plt.xlabel('Importance Score')
plt.show()

importances

**shares**, **likes**, and **views** are the top drivers of popularity score, which matches intuition — the more a meme is shared and liked, the more popular it becomes.

## 10. Conclusion 📝

- We built a regression model to predict a meme's **popularity_score** using engagement metrics (likes, shares, comments, views) and metadata (watch time, caption length, hashtags, posting hour, creator followers).
- **No data leakage** was found — the target is not a direct copy/formula of any single input feature.
- The **XGBoost Regressor** was our best performing model, achieving an **R² score of ~0.99** and a low **MAE (~1.5)**, meaning predictions are, on average, only ~1.5 points away from the true popularity score (on a 0-100 scale).
- **Shares** and **likes** turned out to be the strongest predictors of popularity, followed by **views**.
- This kind of model could help creators/marketers estimate how well a meme might perform before or shortly after posting.

### 🚀 Possible Next Steps
- Try hyperparameter tuning (GridSearchCV / Optuna) to push accuracy even further
- Add more features like meme category/text sentiment if available
- Try ensembling Random Forest + XGBoost predictions

**If you found this notebook helpful, please upvote! 🙌**